# Algo Ninja — Exploration Notebook

Scratch space for EDA, indicator visualization, and RF hyperparameter tuning.
Run `main.py` for the full production pipeline — use this notebook for ad-hoc exploration.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from data.data_loader import load_csv, train_test_split_series
from indicators.technical_indicators import add_all_indicators
from models.rf_predictor import RandomForestPredictor, RFPredictorConfig
from strategies.strategy_engine import STRATEGY_REGISTRY
from backtest.backtester import Backtester, run_all_strategies

In [ ]:
raw = load_csv('../data/sample_ohlcv.csv')
featured = add_all_indicators(raw)
featured[['Close', 'EMA7', 'RSI8', 'ADX9']].tail(60).plot(subplots=True, figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
train, test = train_test_split_series(featured, test_size=0.25)
model = RandomForestPredictor(RFPredictorConfig(n_estimators=300, max_depth=8))
model.fit(train)
print(model.evaluate(test))
model.feature_importances().plot(kind='barh')
plt.title('RF Feature Importances')
plt.show()

In [ ]:
test = test.copy()
test['predicted_return'] = model.predict(test)

bt = Backtester()
leaderboard = run_all_strategies(test, STRATEGY_REGISTRY, bt)
leaderboard

In [ ]:
best_name = leaderboard.iloc[0]['strategy']
result = bt.run(test, STRATEGY_REGISTRY[best_name], best_name)
result.equity_curve.plot(title=f'Equity Curve — {best_name}', figsize=(10, 4))
plt.show()